In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
from math import radians, cos, sin, asin, sqrt

In [ ]:
# 1. Load DataFrame
df = pd.read_csv('C:\\Users\\liad1\\OneDrive\\מסמכים\\personal git\\work_assignments\\cyber_1\\intelos_task_detect_vpn.csv',
                 parse_dates=['REQUEST_TIME'], dayfirst=False)
df = df.sort_values(by='REQUEST_TIME').reset_index(drop=True)
df.head()

,REQUEST_TIME,DEVICE_ID,GEO_LAT,GEO_LON,IP
0,2023-11-13 18:41:00,1,50.603800,5.410800,81.242.16.58
1,2024-01-25 20:12:00,1,50.851426,4.378300,81.242.37.73
2,2024-02-23 14:58:00,1,50.520600,5.547700,81.243.5.66
3,2024-03-03 18:15:00,1,50.728650,4.430460,81.244.47.131
4,2024-03-16 22:01:00,1,50.056301,4.312722,103.86.99.40


In [ ]:
# 2. Haversine function to calculate distance between two coordinates
def haversine(lat1, lon1, lat2, lon2):
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371 # Radius of earth in kilometers
    return c * r


In [ ]:
# 3. Calculate Speed and Flags
df['SPEED_KMH'] = 0.0
df['CLASSIFICATION'] = 'Non-VPN' # Default baseline

for i in range(1, len(df)):
    # Calculate time difference in hours
    time_diff = (df.loc[i, 'REQUEST_TIME'] - df.loc[i-1, 'REQUEST_TIME']).total_seconds() / 3600.0
    df.loc[i, 'TIME_DIFF_HOURS'] = time_diff

    # Calculate distance in km
    dist = haversine(df.loc[i-1, 'GEO_LAT'], df.loc[i-1, 'GEO_LON'], 
                     df.loc[i, 'GEO_LAT'], df.loc[i, 'GEO_LON'])
    df.loc[i, 'DIST_KM'] = dist

    if time_diff > 0:
        speed = dist / time_diff
        df.loc[i, 'SPEED_KMH'] = speed
        
        # If speed > 900 km/h (impossible physical travel)
        if speed > 900:
            # Mark the anomalous IP as VPN
            df.loc[i, 'CLASSIFICATION'] = 'VPN'
            
            # Retroactively check if the previous IP was the baseline or a VPN
            if df.loc[i-1, 'IP'] != df.loc[i, 'IP']:
                 df.loc[i-1, 'CLASSIFICATION'] = 'Uncertain'

print(df[['REQUEST_TIME', 'IP', 'GEO_LAT', 'GEO_LON', 'SPEED_KMH', 'CLASSIFICATION']])

In [17]:
i=0
df.loc[i,'domain']
for ip in df['IP'].unique():
    print(ip)
    print(socket.gethostbyaddr(ip)[0])

81.242.16.58
58.16-242-81.adsl-dyn.isp.belgacom.be
81.242.37.73
73.37-242-81.adsl-dyn.isp.belgacom.be
81.243.5.66
66.5-243-81.adsl-static.isp.belgacom.be
81.244.47.131
131.47-244-81.adsl-dyn.isp.belgacom.be
103.86.99.40


herror: [Errno 11004] host not found

In [ ]:
#bouns -- ways to improve
    #find domain per ip ( if domain include VPN or not)
import socket
i=-1
df['domain'] = "Unknown"

for ip in df['IP'].unique():
    i= i+1
    try:
        domain = socket.gethostbyaddr(ip)[0]
        df.loc[i,'domain'] = domain
        #print(f"IP: {ip}, Domain: {domain[0]}")
    except socket.herror:
        df.loc[i,'domain'] = "Unknown"

In [20]:
df.domain.unique

<bound method Series.unique of 0         58.16-242-81.adsl-dyn.isp.belgacom.be
1         73.37-242-81.adsl-dyn.isp.belgacom.be
2       66.5-243-81.adsl-static.isp.belgacom.be
3        131.47-244-81.adsl-dyn.isp.belgacom.be
4                                       Unknown
5        135.59-241-81.adsl-dyn.isp.belgacom.be
6        177.43-241-81.adsl-dyn.isp.belgacom.be
7                      104-238-169-5.choopa.net
8        230.27-243-81.adsl-dyn.isp.belgacom.be
9        218.24-245-81.adsl-dyn.isp.belgacom.be
10        166.9-245-81.adsl-dyn.isp.belgacom.be
11       229.55-245-81.adsl-dyn.isp.belgacom.be
12      90.0-246-81.adsl-static.isp.belgacom.be
13                                      Unknown
14        57.37-245-81.adsl-dyn.isp.belgacom.be
15                                      Unknown
16                                      Unknown
17        77.26-242-81.adsl-dyn.isp.belgacom.be
18        19.34-240-81.adsl-dyn.isp.belgacom.be
19          host-185-77-76-18.redgeguardian.com
20      2